pip install requests wikitextparser re os json

---
# Data Extraction from Terraria Wiki

In [1]:
import requests
import wikitextparser as wtp
import re
import os
import json
from kb_extraction import get_clean_terraria_wiki
from json_extract import save_to_manifest

## Guide:Getting Started

In [2]:
page_title = "Guide:Getting_started"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Getting_started"
game_state = "Pre-Hardmode"

In [3]:
with open("getting_started.txt", "w", encoding="utf-8") as file:
    file.write(content)

In [4]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_getting_started.json


## Crafting 101

In [5]:
page_title = "Guide:Crafting_101"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Crafting_101"
game_state = "Pre-Hardmode"

In [6]:
with open("crafting101.txt", "w") as file:
    file.write(content)

In [7]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_crafting_101.json


## Class Setups

In [8]:
page_title = "Guide:Class_setups"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Class_setups"
game_state = "Pre-Hardmode"

In [9]:
with open("class_setups.txt", "w") as file:
    file.write(content)

In [10]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_class_setups.json


## Pre-Hardmode Walkthrough

In [11]:
page_title = "Guide:Walkthrough"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Walkthrough"
game_state = "Pre-Hardmode"

In [12]:
with open("Pre-Hardmode_Walkthrough.txt", "w") as file:
    file.write(content)

In [13]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_walkthrough.json


## Getting started with hardmode

In [14]:
page_title = "Guide:Getting_started_with_Hardmode"
content = get_clean_terraria_wiki(page_title)
source_url = "https://terraria.wiki.gg/wiki/Guide:Getting_started_with_Hardmode"
game_state = "Hardmode"

In [15]:
with open("Getting_started_with_Hardmode.txt", "w") as file:
    file.write(content)

In [16]:
save_to_manifest(page_title, content, source_url, game_state)

Successfully saved manifest to: knowledge_base/processed/guide_getting_started_with_hardmode.json


# Indexing

In [17]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader

folder_path = "./knowledge_base/processed"
loader = DirectoryLoader(
    path=folder_path, 
    glob="*.json",
    loader_cls=JSONLoader,
    loader_kwargs = {
        "jq_schema": ".content",
        "text_content": False
    }
)

docs = loader.load()

print(f"Loaded {len(docs)} files.")
print(f"Snippet of first file: {docs[0].page_content[:50]}...")

Loaded 5 files.
Snippet of first file: Terraria has no formal player class or leveling sy...


In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print(f"Number of chunks created: {len(chunks)}")
print()

print("--- Chunk 0 (inspect before storing!) ---")
print(chunks[0].page_content)
print()
print("--- Chunk 1 ---")
print(chunks[1].page_content)

Number of chunks created: 512

--- Chunk 0 (inspect before storing!) ---
Terraria has no formal player class or leveling system. However, weapons can be grouped into four distinct categories based on their damage type – melee, ranged, magic, and summoner. Alongside weapons, several armor sets and accessories operate in tandem to these and potentially can only benefit specific damage types and/or playstyles. Each class has its strengths and weaknesses and has a wide variety of weapons to choose from.

--- Chunk 1 ---
; Melee
The melee class sports the highest defense across progression and decent crowd control, but many melee weapons are limited in range. Although short-ranged melee weapons tend to deal more damage and are less of a challenge to use with the class' high defense, those that fire projectiles are favored especially later in progression where the former weapons require specialized tank builds to use effectively.


In [19]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("Embedding model loaded: all-MiniLM-L6-v2 (384-dimensional)")
print("Embedding all chunks into the vector database...")
db = FAISS.from_documents(chunks, embeddings)

print(f"Vector database created with {len(chunks)} vectors")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_21512\3408347235.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded: all-MiniLM-L6-v2 (384-dimensional)
Embedding all chunks into the vector database...
Vector database created with 512 vectors


In [20]:
db.save_local("faiss_index")
print("FAISS index saved to 'faiss_index/'")
print()
print("To reload later (skip the embedding step entirely):")
print("  db = FAISS.load_local('faiss_index', embeddings, allow_dangerous_deserialization=True)")

FAISS index saved to 'faiss_index/'

To reload later (skip the embedding step entirely):
  db = FAISS.load_local('faiss_index', embeddings, allow_dangerous_deserialization=True)


In [21]:
retriever = db.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created (k=3)")

Retriever created (k=3)


In [22]:
test_query = "How to craft items?"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Number of chunks retrieved: {len(retrieved_docs)}")
print()
for i, doc in enumerate(retrieved_docs):
    print(f"--- Retrieved Chunk {i+1} ---")
    print(doc.page_content)
    print()

Query: How to craft items?
Number of chunks retrieved: 3

--- Retrieved Chunk 1 ---
Crafting is the act of combining one or more materials into a different item, usually at a specific crafting station. The crafting system is deeply interwoven with game progression, with many "key" materials and crafting stations dropped by various bosses, or otherwise available only after a given boss has been defeated.

--- Retrieved Chunk 2 ---
The crafting menu will show the player possible Recipes they can craft with the materials they have in their current position. With wood, you can craft a Work Bench and Wood Platform. If you killed a Slime and have Gel, you can also craft Torch. The Work Bench is used as a Crafting Station to craft more new items, Platforms are blocks you can stand on but also move through, and Torches produce light to see in dark areas. All of these items can be placed; the Work Bench must be placed on top of

--- Retrieved Chunk 3 ---
To navigate this menu, you can click or 